# Predictive Maintenance — Failure Sensor Predictor

AVIP 2026 Task 8 workflow notebook.

This notebook mirrors the production pipeline in `src/`:
1. Dataset loading
2. Inspection & cleaning
3. Timestamp / failure analysis
4. Feature engineering (12h causal rolling)
5. Chronological split
6. Model training & evaluation

**Run from the project root** (or ensure the project root is on `sys.path`).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from src.data_loader import load_labeled_dataset, load_raw_tables, dataset_summary
from src.feature_engineering import engineer_features, chronological_split, get_feature_matrix
from src.config import PREDICTION_HORIZON_HOURS, ROLLING_WINDOW_HOURS

sns.set_theme(style="whitegrid")
print("Project root:", ROOT)
print("Horizon hours:", PREDICTION_HORIZON_HOURS)
print("Rolling window hours:", ROLLING_WINDOW_HOURS)

## 1–2. Load & inspect raw data

In [ ]:
tables = load_raw_tables()
telemetry, failures, machines, errors = (
    tables["telemetry"], tables["failures"], tables["machines"], tables["errors"]
)
print("Telemetry:", telemetry.shape)
display(telemetry.head())
print("Failures:", failures.shape)
display(failures.head())
display(telemetry.describe())

## 3–5. Cleaning, timestamps, failure-event analysis & labels

In [ ]:
labeled = load_labeled_dataset()
print(dataset_summary(labeled))
print("Null counts:\n", labeled.isna().sum())

fig, ax = plt.subplots(figsize=(6, 3))
labeled["will_fail_within_horizon"].value_counts().sort_index().plot(
    kind="bar", ax=ax, color=["#4C78A8", "#E45756"]
)
ax.set_title("Target: failure within next 24 hours")
ax.set_xticklabels(["0", "1"], rotation=0)
plt.show()

print("Failures by component:\n", failures["failure"].value_counts())

## 6. Feature engineering (causal 12-hour rolling)

In [ ]:
engineered = engineer_features(labeled, window_hours=ROLLING_WINDOW_HOURS)
feature_cols = engineered.attrs["feature_cols"]
print(f"Rows after rolling dropna: {len(engineered):,}")
print(f"n_features: {len(feature_cols)}")
print(feature_cols)

mid = int(engineered.loc[engineered["will_fail_within_horizon"] == 1, "machineID"].mode().iloc[0])
sample = engineered[engineered["machineID"] == mid].tail(24 * 14)
plt.figure(figsize=(10, 3))
plt.plot(sample["datetime"], sample["vibration_current"], label="vibration")
plt.plot(sample["datetime"], sample["vibration_roll_mean_12h"], label="12h mean")
plt.legend(); plt.title(f"Machine {mid} — rolling vibration"); plt.show()

## 7. Time-aware train / validation / test split

In [ ]:
splits = chronological_split(engineered)
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]
print("train", train_df["datetime"].min(), "→", train_df["datetime"].max(), len(train_df))
print("val  ", val_df["datetime"].min(), "→", val_df["datetime"].max(), len(val_df))
print("test ", test_df["datetime"].min(), "→", test_df["datetime"].max(), len(test_df))
assert train_df["datetime"].max() <= val_df["datetime"].min()
assert val_df["datetime"].max() <= test_df["datetime"].min()
print("Chronological order OK (no overlap).")

## 8–11. Train model, evaluate, confusion matrix, feature importance

For a full training run that also writes `models/` and `reports/`, execute:

```bash
python src/train.py
```

Below we load the saved artifact if present, otherwise train a compact model inline.

In [ ]:
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

model_path = ROOT / "models" / "failure_predictor.joblib"
X_train, y_train, _ = get_feature_matrix(train_df, feature_cols)
X_val, y_val, _ = get_feature_matrix(val_df, feature_cols)
X_test, y_test, _ = get_feature_matrix(test_df, feature_cols)

if model_path.exists():
    artifact = joblib.load(model_path)
    model = artifact["model"]
    threshold = float(artifact["threshold"])
    print("Loaded saved model. threshold=", threshold)
else:
    sw = compute_sample_weight("balanced", y_train)
    model = HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.08, max_iter=150,
        min_samples_leaf=50, l2_regularization=0.1,
        early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=12, random_state=42,
    )
    model.fit(X_train, y_train, sample_weight=sw)
    val_prob = model.predict_proba(X_val)[:, 1]
    threshold = 0.5
    best = -1
    for t in np.linspace(0.2, 0.8, 25):
        pred = (val_prob >= t).astype(int)
        score = 0.4 * precision_score(y_val, pred, zero_division=0) + 0.6 * recall_score(y_val, pred, zero_division=0)
        if score > best:
            best, threshold = score, float(t)
    print("Trained inline. threshold=", threshold)

prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= threshold).astype(int)
print(classification_report(y_test, pred, digits=4, zero_division=0))
print("Accuracy", accuracy_score(y_test, pred))
print("Precision", precision_score(y_test, pred, zero_division=0))
print("Recall", recall_score(y_test, pred, zero_division=0))
print("F1", f1_score(y_test, pred, zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["0", "1"]).plot(ax=ax, cmap="Blues")
ax.set_title("Test confusion matrix")
plt.show()

In [ ]:
rng = np.random.RandomState(42)
idx = rng.choice(len(X_val), size=min(3000, len(X_val)), replace=False)
imp = permutation_importance(
    model, X_val.iloc[idx], y_val.iloc[idx],
    n_repeats=3, random_state=42, scoring="f1", n_jobs=-1
)
order = np.argsort(imp.importances_mean)[-12:]
plt.figure(figsize=(8, 5))
plt.barh(np.array(feature_cols)[order], imp.importances_mean[order])
plt.title("Permutation importance (val subsample)")
plt.tight_layout(); plt.show()

## 12. Example predictions

In [ ]:
examples = test_df.copy()
examples["prob"] = prob
examples["pred"] = pred
print("High-risk examples (predicted=1):")
display(examples[examples["pred"] == 1][
    ["datetime", "machineID", "volt", "rotate", "pressure", "vibration", "will_fail_within_horizon", "prob"]
].head(10))
print("Low-risk examples (predicted=0):")
display(examples[examples["pred"] == 0][
    ["datetime", "machineID", "volt", "rotate", "pressure", "vibration", "will_fail_within_horizon", "prob"]
].head(10))